In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----- Hyperparameters -----
SEQ_LEN = 8
PRED_LEN = 12
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_LAYERS = 1
EPOCHS = 10
LR = 0.001

# ----- Load and Preprocess Data -----
df = pd.read_csv("converted_zara_1.csv")

# Normalize coordinates
x_mean, x_std = df['x'].mean(), df['x'].std()
y_mean, y_std = df['y'].mean(), df['y'].std()
df['x'] = (df['x'] - x_mean) / x_std
df['y'] = (df['y'] - y_mean) / y_std

# Group by person_id and create sequences
trajectories = []
for pid, person_df in df.groupby("person_id"):
    person_df = person_df.sort_values("frame_id")
    coords = person_df[['x', 'y']].values
    for i in range(len(coords) - SEQ_LEN - PRED_LEN):
        obs = coords[i:i+SEQ_LEN]
        pred = coords[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN]
        trajectories.append((obs, pred))

# ----- Dataset Definition -----
class TrajectoryDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        obs, fut = self.data[idx]
        return torch.tensor(obs, dtype=torch.float32), torch.tensor(fut, dtype=torch.float32)

# ----- LSTM Model -----
class LSTMModel(nn.Module):
    def __init__(self, input_size=2, hidden_size=HIDDEN_SIZE, output_len=PRED_LEN):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, NUM_LAYERS, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2 * output_len)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = self.fc(hn[-1])
        return out.view(-1, PRED_LEN, 2)

# ----- Training Loop -----
dataset = TrajectoryDataset(trajectories)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for obs, fut in dataloader:
        pred = model(obs)
        loss = criterion(pred, fut)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(dataloader):.4f}")

# ----- Save Model -----
torch.save(model.state_dict(), "lstm_zara_model.pth")


Epoch 1/10, Loss: 0.0591
Epoch 2/10, Loss: 0.0057
Epoch 3/10, Loss: 0.0027
Epoch 4/10, Loss: 0.0015
Epoch 5/10, Loss: 0.0011
Epoch 6/10, Loss: 0.0008
Epoch 7/10, Loss: 0.0006
Epoch 8/10, Loss: 0.0005
Epoch 9/10, Loss: 0.0005
Epoch 10/10, Loss: 0.0004
